# 03 — Run Retrieval Experiments

This notebook runs all three retrieval strategies against the 40-question benchmark and saves the results.

**Strategies tested:**
- `vanilla`: Pure cosine similarity, no filters
- `metadata_filtered`: Pre-filter by ticker + year, then similarity search
- `task_aware`: Routes to split-budget retrieval for cross-year/cross-company tasks

**Output:** `data/processed/retrieval_results.parquet` — long-format table with one row per (question, strategy, retrieved_chunk).

**Runtime:** ~3–5 minutes (120 queries × 3 strategies = 360 embedding calls).

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd

from src.config import PROCESSED_DIR, TOP_K
from src.retrieval import (
    get_qdrant_client,
    vanilla_retrieval,
    metadata_filtered_retrieval,
    task_aware_retrieval,
    run_retrieval_experiments,
)

questions_path = os.path.join(PROCESSED_DIR, 'questions.csv')
questions_df = pd.read_csv(questions_path)
print(f"Loaded {len(questions_df)} questions")
print(questions_df['task_type'].value_counts())

## 3.1 — Connect to Qdrant and verify collection

In [ ]:
from src.config import QDRANT_COLLECTION

client = get_qdrant_client()
info = client.get_collection(QDRANT_COLLECTION)
print(f"Collection: {QDRANT_COLLECTION}")
print(f"  Points: {info.points_count}")
print(f"  Vectors: {info.vectors_count}")

if info.points_count == 0:
    raise RuntimeError("Collection is empty! Run notebook 02 first to upsert chunks.")

## 3.2 — Single question demo

Compare how the three strategies retrieve different context for the same question.

In [ ]:
demo_q = questions_df[questions_df['task_type'] == 'cross_year'].iloc[0]
question = demo_q['question']
ticker = demo_q['ticker']
year = int(demo_q['year'])
comp_year = int(demo_q['comparison_year']) if pd.notna(demo_q['comparison_year']) else None

print(f"Question: {question}")
print(f"Expected: {ticker} | years: {year} vs {comp_year}")
print("\n" + "="*80)

vanilla_results = vanilla_retrieval(client, question, top_k=TOP_K)
filtered_results = metadata_filtered_retrieval(client, question, ticker=ticker, year=year, top_k=TOP_K)
task_results = task_aware_retrieval(
    client, question,
    task_type='cross_year',
    companies=[ticker],
    years=[year, comp_year],
    top_k=TOP_K
)

for strategy, results in [('vanilla', vanilla_results), ('metadata_filtered', filtered_results), ('task_aware', task_results)]:
    print(f"\n--- {strategy.upper()} ---")
    tickers_retrieved = [r['ticker'] for r in results]
    years_retrieved = [r['year'] for r in results]
    print(f"  Tickers: {tickers_retrieved}")
    print(f"  Years:   {years_retrieved}")
    for r in results:
        print(f"  [{r['ticker']} {r['year']} {r['section']}] score={r['score']:.3f}")

## 3.3 — Run full retrieval experiment

This runs all 40 questions × 3 strategies = 120 retrieval calls.

In [ ]:
output_path = os.path.join(PROCESSED_DIR, 'retrieval_results.parquet')

if os.path.exists(output_path):
    print(f"Loading existing results from {output_path}")
    retrieval_df = pd.read_parquet(output_path)
else:
    print("Running retrieval experiments...")
    retrieval_df = run_retrieval_experiments(questions_df, client=client)
    retrieval_df.to_parquet(output_path, index=False)
    print(f"Saved to {output_path}")

print(f"\nShape: {retrieval_df.shape}")
print(f"Strategies: {retrieval_df['strategy'].unique()}")
retrieval_df.head()

## 3.4 — Retrieval quality analysis

For questions with a specific expected ticker and year, compute how often the correct entity appears in the top-5.

In [ ]:
# Merge question metadata into retrieval results
merged = retrieval_df.merge(
    questions_df[['question_id', 'task_type', 'ticker', 'year']].rename(
        columns={'ticker': 'expected_ticker', 'year': 'expected_year'}
    ),
    on='question_id',
    how='left'
)

# Entity hit rate: expected ticker appears in retrieved chunks
def entity_hit(group, expected_ticker_col='expected_ticker'):
    expected = group[expected_ticker_col].iloc[0]
    return expected in group['ticker'].values

hit_rate = (
    merged[merged['expected_ticker'].notna()]
    .groupby(['question_id', 'strategy', 'task_type', 'expected_ticker'])
    .apply(entity_hit)
    .reset_index(name='entity_hit')
)

print("Entity hit rate by strategy:")
print(hit_rate.groupby('strategy')['entity_hit'].mean().round(3))

print("\nEntity hit rate by strategy × task_type:")
print(hit_rate.groupby(['strategy', 'task_type'])['entity_hit'].mean().unstack().round(3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

hit_pivot = hit_rate.groupby(['strategy', 'task_type'])['entity_hit'].mean().reset_index()
hit_pivot = hit_pivot.pivot(index='strategy', columns='task_type', values='entity_hit')

plt.figure(figsize=(9, 4))
sns.heatmap(hit_pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1)
plt.title('Entity Retrieval Hit Rate by Strategy × Task Type')
plt.tight_layout()
plt.savefig('../figures/retrieval_hit_rate.png', dpi=150, bbox_inches='tight')
plt.show()